In [1]:
import math
import torch
import torch.nn as nn

/Users/aslonkhamidov/Desktop/code/learning/deep_learning_mit/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
class EmbeddingWithProjection(nn.Module):
    def __init__(self, vocab_size, d_embed, d_model, max_position_embeddings=512, dropout=0.1) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_embed = d_embed
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(self.vocab_size, self.d_embed)
        self.projection = nn.Linear(self.d_embed, self.d_model)
        self.scaling = float(math.sqrt(self.d_model))
        
        
        self.layernorm = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(p=dropout)
        
        
    @staticmethod
    def create_positional_encoding(seq_length, d_model, batch_size=1):
        # create position indices: [seq_lenfth, 1]
        position = torch.arange(seq_length).unsqueeze(1).float()
        
        
        # create dimention indices: [1, d_model // 2]
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10_000) / d_model)
        )
        
        # create empty tensor: [seq_length, d_model]
        pe = torch.zeros(seq_length, d_model)
        
        # compute sine and cosine
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # add batch dimension and expand: [batch_size, seq_length, d_model]
        pe = pe.unsqueeze(0).expand(batch_size, -1, -1)
        
        return pe
        

    def forward(self, x: torch.Tensor):
        assert x.dtype == torch.long, f"Input tensor dtype mismatch: should torch.long"
        batch_size, seq_length = x.size() # [batch, seq_length]
        
        
        # token embedding
        token_embedding = self.embedding(x) #[2, 16, 1024]     
        
        
        # project the scaled token embedding to the d_model space
        token_embedding = self.projection(token_embedding) * self.scaling  #[2, 16, 768]
        
        # add positional encodings to projected,
        # scaled embeddings before applying layer norm and dropout. 
        positional_encoding = self.create_positional_encoding(seq_length, self.d_model, batch_size)  #[2, 16, 768]
        
        # in addition, we apply dropout to the sums of the embeddings
        # in both the encoder and decoder stacks. for the base model, we use arate of Pdrop = 0.1 
        normalized_sum = self.layernorm(token_embedding + positional_encoding)
        final_output = self.dropout(normalized_sum)
        
        
        return final_output
        
        
        

In [3]:
from typing import Any
import torch.nn.functional as F


class TransformerAttention(nn.Module):
    """
    """
    
    def __init__(self, d_model, num_head, dropout=0.1, bias=True,  *args: Any, **kwargs: Any) -> None:
        super().__init__()
        assert d_model % num_head == 0, "d_model must be divisible by num_head"
        
        self.d_model = d_model
        self.num_head = self.num_head
        self.d_head = d_model // num_head
        self.dropout_rate = dropout # store dropout rate specifically
        
        
        # linear transformations 
        self.q_proj = nn.Linear(d_model, d_model, bias=bias)
        self.k_proj = nn.Linear(d_model, d_model, bias=bias)
        self.v_proj = nn.Linear(d_model, d_model, bias=bias)
        self.output_proj = nn.Linear(d_model, d_model, bias=bias)
        
        # Dropout Layer
        self.dropout = nn.Dropout(p=dropout)
        
        
        # initialize scaler
        self.scaler = float(1.0 / math.sqrt(self.d_head)) # store as float in initialization
        
        
    def forward(self, sequence, key_value_states = None, att_mask = None):
        batch_size, seq_len, model_dim = sequence.size()
        
        # check only ciritical input dimestions 
        assert model_dim == self.d_model, f"Input dimension {model_dim} doesn't match model dimension {self.d_model}"
        
        if key_value_states is not None:
            assert key_value_states.size(-1) == self.d_model, f"error"
            
            
        # if key_value_states are provided this layer is used as cross-attention layer
        # for the decoder
        is_cross_attention = key_value_states  is not None
        
        # Linear projection and reshape for multi_head
        Q_state = self.q_proj(sequence)
        if is_cross_attention:
            kv_seq_len = key_value_states.size(1)
            K_state = self.k_proj(key_value_states)
            V_state = self.v_proj(key_value_states)
            
        else:
            kv_seq_len = sequence
            K_state = self.k_proj(sequence)
            V_state = self.v_proj(sequence)
            
        # [batch_size, self.num_head, seq_len, self.d_head]
        Q_state = Q_state.view(batch_size, seq_len, self.num_head, self.d_head).transpose(1,2)
        
        # in cross attention , key/value csequence lengtrh might be different from query sequence length
        K_state = K_state.view(batch_size, seq_len, self.num_head, self.d_head).transpose(1,2)
        V_state = V_state.view(batch_size, seq_len, self.num_head, self.d_head).transpose(1,2)
        
        # Scale Q by 1/sqrt(d_k)
        Q_state *= self.scaler
        
        # compute attention matrix: QK ^ T
        self.att_matrix = torch.matmul(Q_state, K_state.transpose(-1, -2))
        
        # apply attentio nmas to attention matrix 
        if att_mask is not None and not isinstance(att_mask, torch.Tensor):
            raise Exception("att_mask must be a troch.Tensor")
        
        
        if att_mask is not None:
            self.att_matrix = self.att_matrix + att_mask
            
            
        #apply softmax to the last dimennsion to get attention score: softmax(QK^T)
        att_score = F.softmax(self.att_matrix, dim = -1)
        
        # apply drop out to attention score
        att_score = self.dropout(att_score)
        
        
        # get final output: softmax(QK ^ T) * v
        
        att_output = torch.matmul(att_score, V_state)
        
        # concatinate all attention heads
        att_output = att_output.transpose(1,2)
        att_output = att_output.contiguous().view(batch_size, seq_len, self.num_head * self.d_head)
        
        # final linear transformation to the concatenated output 
        att_output = self.output_proj(att_output)
        
        assert att_output.size() == (batch_size, seq_len, self.d_model) , f"Final output shape {att_output.size()} incorrent"
        
        return att_output
        
            

In [ ]:
from turtle import forward


class FFN(nn.Module):
    
    def __init__(self, d_model, d_ff, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.d_model = d_model
        self.d_ff = d_ff
        
        # linear transformation y = xW + b
        self.fc1 = nn.Linear(self.d_model, self.d_ff, bias = True)
        self.fc2 = nn.Linear(self.d_ff, self.d_model, bias=True)
        
        # for potential speed up
        # pre-normalize the wights (can helpo with training stability)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
    
    def forward(self, input):
        # check input and first FF layer dimension matching 
        batch_size, seq_length, d_input = input.size()
        assert self.d_model == d_input, "d_model must bte the same dimension as the input"
        
        # first linear transformation followed by ReLU
        # there is no need for explicit torch.max() as F.relu() already implements max(0,x)
        f1 = F.relu(self.fc1(input))
        
        # max (0, xW_1 + b_1) * W_2 + b2
        f2 = self.fc2(f1)
        
        return f2
    
        

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(
        self, d_model, d_ff, num_head, dropout=0.1, bias=True, *args: Any, **kwargs: Any
    ) -> None:
        super().__init__(*args, **kwargs)
        self.d_model = d_model
        self.d_ff = d_ff
        
        # attention
        self.att = TransformerAttention(
            d_model=d_model, num_head=num_head, dropout=dropout, bias=bias
        )
        
        # FFN block
        self.ffn = FFN(
            d_model = d_model,
            d_ff = d_ff
        )
        # Dropout layer 
        self.dropout = nn.Dropout(p=dropout)
        
        # Layer normalization layer
        self.LayerNorm_att = nn.LayerNorm(self.d_model)
        self.LayerNorm_fnn = nn.LayerNorm(self.d_model)
        
        
    def forward(self, embed_input, padding_mask = None):
        batch_size, seq_len, _ = embed_input.size()
        
        
        # First sublayer: self attention
        
        att_sublayer = self.att(sequence = embed_input, key_value_states = None, att_mask = padding_mask) # [ batch_size, sequence_length,d_model]
        
        # apply dropout before layer normalization for each sublayer
        att_sublayer = self.dropout(att_sublayer)
        # residual layer normalization 
        att_normalized  = self.LayerNorm_att(embed_input + att_sublayer) # # [ batch_size, sequence_length,d_model]
        
        # Second sublayer: FNN
        ffn_sublayer = self.ffn(att_normalized)
        ffn_sublayer = self.dropout(ffn_sublayer)
        ffn_normalized = self.LayerNorm_fnn(att_normalized +  ffn_normalized)
        return ffn_normalized
        
        